# Register migrated tables in the us-east-1 catalog

**Run in JupyterHub on the us-east-1 cluster.** Part of RTIInternational/teehr-hub#465.

Prerequisites: `02_rewrite_table_paths` ran on us-east-2, `03_copy_staged_metadata.sh` reported every table
ready, and the run's `rewrite_manifest.json` has reached this cluster's `/data/region_move/` (EFS replication,
or after the EFS failover).

For every table in the manifest this registers the rewritten metadata with `register_table`, then checks that
the current snapshot ID and row count match what `02` recorded on us-east-2.

**Dry run:** set `DRY_RUN = True`. The tables are registered under their real names (Polaris requires a table's
location to be inside its namespace's, so a throwaway namespace can't hold them), checked, and dropped again by
the cleanup cell at the end. Only do that before cutover, while nothing else uses this catalog.

In [ ]:
import json
from pathlib import Path

from teehr.evaluation.spark_session_utils import create_spark_session

In [ ]:
# None picks the newest run under /data/region_move.
RUN_ID = None

# Dry run: register, check, then drop again with the cleanup cell.
DRY_RUN = False

runs = sorted(p for p in Path("/data/region_move").iterdir() if p.is_dir())
run_dir = Path("/data/region_move") / RUN_ID if RUN_ID else runs[-1]
manifest = json.loads((run_dir / "rewrite_manifest.json").read_text())
print(run_dir, f"{len(manifest['tables'])} tables")

In [ ]:
%%time
spark = create_spark_session()
spark.sql("USE iceberg")

In [ ]:
def metadata_file(entry):
    latest = entry["latest_version"]
    if latest.startswith("s3://"):
        return latest.replace(manifest["source_prefix"], manifest["target_prefix"], 1)
    return f"{entry['target_location']}/metadata/{latest.split('/')[-1]}"


def target_name(fq):
    return fq


for ns in sorted({target_name(e["table"]).split(".")[0] for e in manifest["tables"]}):
    spark.sql(f"CREATE NAMESPACE IF NOT EXISTS iceberg.{ns}")

Register each table.

In [ ]:
registered = []
for entry in manifest["tables"]:
    name = target_name(entry["table"])
    path = metadata_file(entry)
    spark.sql(f"""
        CALL iceberg.system.register_table(
            table => '{name}',
            metadata_file => '{path}'
        )
    """)
    registered.append(name)
    print(f"registered {name} <- {path}")

Check each table against what us-east-2 recorded.

In [ ]:
problems = []
for entry in manifest["tables"]:
    name = target_name(entry["table"])
    snap = spark.sql(
        f"SELECT snapshot_id FROM iceberg.{name}.snapshots ORDER BY committed_at DESC LIMIT 1"
    ).collect()
    snapshot_id = snap[0].snapshot_id if snap else None
    row_count = spark.table(f"iceberg.{name}").count()
    location = next(
        r.data_type
        for r in spark.sql(f"DESCRIBE TABLE EXTENDED iceberg.{name}").collect()
        if r.col_name == "Location"
    )
    ok = (
        snapshot_id == entry["snapshot_id"]
        and row_count == entry["row_count"]
        and location.startswith(manifest["target_prefix"].rstrip("/"))
    )
    print(f"{'ok ' if ok else 'BAD'} {name}: snapshot {snapshot_id}, {row_count:,} rows, {location}")
    if not ok:
        problems.append(name)

print(f"\n{len(problems)} problem(s)" if problems else "\nAll tables match.")

**Dry run only:** drop the registrations again. Without `PURGE` this removes only the catalog entries, not the
files.

In [ ]:
if DRY_RUN:
    for name in registered:
        spark.sql(f"DROP TABLE iceberg.{name}")
    print(f"dropped {len(registered)} dry-run table(s)")

In [ ]:
spark.stop()